# Zebra Finch Call-Type and Age Classification

Classify 11 call types and adult/juvenile age groups in zebra finch (*Taeniopygia guttata*) vocalizations using BEATs and EfficientNet-B0.

**Workflow:** download → explore → embed → UMAP → training-free metrics (NMI, ARI, R-AUC) → linear probe → attention probe (sl-BEATs) → static + interactive figures → save artifacts.

In [ ]:
import pathlib
import sys


def find_repo_root(start: pathlib.Path) -> pathlib.Path:
    for p in [start, *start.parents]:
        if (p / "pyproject.toml").exists():
            return p
    raise FileNotFoundError("Could not locate repo root (pyproject.toml not found).")


REPO_ROOT = find_repo_root(pathlib.Path().resolve())
sys.path.insert(0, str(REPO_ROOT))

import json
import re
import zipfile

import librosa
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import plotly.express as px
import soundfile as sf
import torch
from avex import load_model
from IPython.display import display
from tqdm.auto import tqdm

from utils.probing import compute_training_free_metrics, run_attention_probe, run_linear_probe
from utils.visualization import (
    confusion_heatmap_static,
    plot_model_comparison,
    plot_model_comparison_static,
    plot_umap,
    plot_umap_static,
)

EXAMPLE_DIR = REPO_ROOT / "examples" / "05_zebra_finch"
DATA_DIR = EXAMPLE_DIR / "data"
AUDIO_DIR = DATA_DIR / "audio"
EMBED_DIR = DATA_DIR / "embeddings"
EMBED_DIR.mkdir(parents=True, exist_ok=True)

TARGET_SR = 16_000
DEVICE = "cpu"
BEATS_MODEL = "esp_aves2_sl_beats_all"
EFFNET_MODEL = "esp_aves2_effnetb0_all"
print("Setup complete.")

## 1. Download Dataset

In [ ]:
FIGSHARE_ARTICLE_ID = 11905533
FIGSHARE_API_URL = f"https://api.figshare.com/v2/articles/{FIGSHARE_ARTICLE_ID}/files"
ARCHIVE_PATH = DATA_DIR / "zebra_finch_calls.zip"
EXTRACT_SENTINEL = AUDIO_DIR / ".extracted"
import json as _json_mod
import urllib.request as _ur


def download_with_progress(url, dest):
    def _hook(count, bs, total):
        if total > 0:
            print(f"\r  {min(count * bs / total * 100, 100):.1f}%", end="", flush=True)

    _ur.urlretrieve(url, dest, reporthook=_hook)
    print()


AUDIO_DIR.mkdir(parents=True, exist_ok=True)
if not ARCHIVE_PATH.exists():
    with _ur.urlopen(FIGSHARE_API_URL) as resp:
        files_info = _json_mod.loads(resp.read())
    target = max(files_info, key=lambda f: f["size"])
    print(f"Downloading {target['name']} ({target['size'] / 1e6:.0f} MB) ...")
    download_with_progress(target["download_url"], ARCHIVE_PATH)
else:
    print(f"Archive present ({ARCHIVE_PATH.stat().st_size / 1e6:.1f} MB) — skipping.")

if not EXTRACT_SENTINEL.exists():
    print("Extracting ...")
    with zipfile.ZipFile(ARCHIVE_PATH) as zf:
        zf.extractall(AUDIO_DIR)
    EXTRACT_SENTINEL.touch()
    print("Done.")
else:
    print("Audio already extracted.")
print(f"WAV files: {len(list(AUDIO_DIR.glob('**/*.wav')))}")

## 2. Data Exploration

In [ ]:
CALL_TYPE_NAMES = {
    "aggc": "aggressive call",
    "ag": "aggressive call",
    "dc": "distance call",
    "disc": "distance call",
    "ltc": "long-term call",
    "nekakle": "nekakle call",
    "nekaklec": "nekakle call",
    "nestc": "nest call",
    "nest": "nest call",
    "ne": "nest call",
    "nearkc": "nest call",
    "neseq": "nest call seq",
    "nestseq": "nest call seq",
    "nestcseq": "nest call seq",
    "so": "song",
    "song": "song",
    "tetc": "tet call",
    "tet": "tet call",
    "te": "tet call",
    "thuckc": "thuck call",
    "thukc": "thuck call",
    "thuc": "thuck call",
    "tukc": "thuck call",
    "whinec": "whine call",
    "whine": "whine call",
    "whi": "whine call",
    "wh": "whine call",
    "wc": "whine call",
    "whic": "whine call",
    "whinecseq": "whine call seq",
    "whicnestc": "whine+nest call",
    "beggseq": "begging seq",
}
META_PATH = DATA_DIR / "metadata.csv"
FNAME_RE = re.compile(r"^([A-Za-z][A-Za-z0-9]*)_\d*-([A-Za-z]+)-", re.IGNORECASE)

if META_PATH.exists():
    # Load existing metadata (preserves same row ordering as any cached embeddings)
    df = pd.read_csv(META_PATH)
    if "call_name" not in df.columns and "call_type" in df.columns:
        df["call_name"] = df["call_type"].map(lambda c: CALL_TYPE_NAMES.get(c, c))
    if "age" not in df.columns:
        df["age"] = "unknown"
    print(f"Loaded existing metadata: {len(df)} rows")
else:
    wav_files = sorted(AUDIO_DIR.glob("**/*.wav"))
    records = []
    for wav_path in wav_files:
        m = FNAME_RE.search(wav_path.name)
        if m is None:
            continue
        specimen = m.group(1)
        call_code = m.group(2).lower()
        info = sf.info(str(wav_path))
        records.append(
            {
                "path": str(wav_path),
                "filename": wav_path.name,
                "specimen": specimen,
                "call_type": call_code,
                "call_name": CALL_TYPE_NAMES.get(call_code, call_code),
                "age": "unknown",
                "duration_s": info.duration,
                "sample_rate": info.samplerate,
            }
        )
    df = pd.DataFrame(records)
    df.to_csv(META_PATH, index=False)
    print(f"Parsed and saved: {len(df)} files")

print(f"Call types: {df['call_name'].nunique()}, specimens: {df['specimen'].nunique()}")

In [ ]:
ct_df = df.groupby("call_name").size().reset_index(name="count").sort_values("count", ascending=False)
fig = px.bar(
    ct_df,
    x="call_name",
    y="count",
    title="Zebra finch call type counts",
    labels={"call_name": "Call type", "count": "Number of calls"},
    color="count",
    color_continuous_scale="Blues",
)
fig.update_layout(xaxis_tickangle=-30, showlegend=False)
fig.show()

## 3. Embedding Extraction

In [ ]:
MIN_SAMPLES = int(0.5 * TARGET_SR)


def load_audio(path: str, target_sr: int = TARGET_SR) -> torch.Tensor:
    """Load a WAV, convert to mono, resample, zero-pad to at least 0.5 s."""
    wav, sr = sf.read(path, dtype="float32", always_2d=True)
    wav = wav.mean(axis=1)
    if sr != target_sr:
        wav = librosa.resample(wav, orig_sr=sr, target_sr=target_sr)
    if len(wav) < MIN_SAMPLES:
        wav = np.pad(wav, (0, MIN_SAMPLES - len(wav)))
    return torch.from_numpy(wav).unsqueeze(0)


print(f"Sample shape: {load_audio(df['path'].iloc[0]).shape}")

In [ ]:
BEATS_CACHE = EMBED_DIR / "beats_embeddings.npy"

if BEATS_CACHE.exists() and np.load(BEATS_CACHE).shape[0] == len(df):
    beats_embs = np.load(BEATS_CACHE)
    print(f"Loaded cached BEATs last-layer embeddings: {beats_embs.shape}")
else:
    print(f"Loading model: {BEATS_MODEL}")
    model = load_model(BEATS_MODEL, return_features_only=True, device=DEVICE)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for path in tqdm(df["path"], desc="BEATs last-layer"):
            wav = load_audio(path)
            feats = model(wav)  # (1, T, 768)
            embeddings.append(feats.mean(dim=1).squeeze(0).cpu().numpy())
    beats_embs = np.stack(embeddings)
    np.save(BEATS_CACHE, beats_embs)
    print(f"Saved BEATs last-layer embeddings: {beats_embs.shape}")
    del model

In [ ]:
BEATS_ALL_CACHE = EMBED_DIR / "beats_all_layers_embeddings.npy"

if BEATS_ALL_CACHE.exists() and np.load(BEATS_ALL_CACHE).shape[0] == len(df):
    beats_all_embs = np.load(BEATS_ALL_CACHE)
    print(f"Loaded cached BEATs all-layers embeddings: {beats_all_embs.shape}")
else:
    print(f"Loading model for all-layers extraction: {BEATS_MODEL}")
    model = load_model(BEATS_MODEL, return_features_only=True, device=DEVICE)
    model.eval()

    # Register forward hooks on every transformer encoder layer
    try:
        encoder_layers = model.model.encoder.layers
    except AttributeError:
        encoder_layers = model.backbone.encoder.layers
    n_layers = len(encoder_layers)
    layer_store: dict = {}
    hooks = []
    for i, layer in enumerate(encoder_layers):

        def _make_hook(idx):
            def _hook(module, inp, out):
                layer_store[idx] = out[0] if isinstance(out, tuple) else out

            return _hook

        hooks.append(layer.register_forward_hook(_make_hook(i)))
    print(f"  Registered hooks on {n_layers} transformer layers.")

    all_embs = []
    with torch.no_grad():
        for path in tqdm(df["path"], desc="BEATs all-layers"):
            layer_store.clear()
            wav = load_audio(path)
            _ = model(wav)
            # Mean-pool each layer then average across layers → (D,)
            # Handles both time-first (T, B, D) and batch-first (B, T, D) outputs
            per_layer = [
                layer_store[i].view(-1, layer_store[i].shape[-1]).mean(dim=0).cpu().numpy() for i in range(n_layers)
            ]
            all_embs.append(np.mean(per_layer, axis=0))

    for h in hooks:
        h.remove()

    beats_all_embs = np.stack(all_embs)
    np.save(BEATS_ALL_CACHE, beats_all_embs)
    print(f"Saved BEATs all-layers embeddings: {beats_all_embs.shape}")
    del model

In [ ]:
EFFNET_CACHE = EMBED_DIR / "effnet_embeddings.npy"

if EFFNET_CACHE.exists() and np.load(EFFNET_CACHE).shape[0] == len(df):
    effnet_embs = np.load(EFFNET_CACHE)
    print(f"Loaded cached EfficientNet embeddings: {effnet_embs.shape}")
else:
    print(f"Loading model: {EFFNET_MODEL}")
    model = load_model(EFFNET_MODEL, return_features_only=True, device=DEVICE)
    model.eval()
    embeddings = []
    with torch.no_grad():
        for path in tqdm(df["path"], desc="EfficientNet"):
            wav = load_audio(path)
            feats = model(wav)  # (1, C, H, W)
            embeddings.append(feats.mean(dim=(2, 3)).squeeze(0).cpu().numpy())
    effnet_embs = np.stack(embeddings)
    np.save(EFFNET_CACHE, effnet_embs)
    print(f"Saved EfficientNet embeddings: {effnet_embs.shape}")
    del model

## 4. UMAP Visualisation

In [ ]:
call_labels = df["call_name"].tolist()
hover = [f"{row['filename']}<br>Type: {row['call_name']}" for _, row in df.iterrows()]

fig_beats = plot_umap(
    beats_embs,
    labels=call_labels,
    title=f"UMAP — {BEATS_MODEL} (last layer)<br><sup>colour = call type</sup>",
    hover_text=hover,
)
fig_beats.show()
fig_umap_beats_static = plot_umap_static(
    beats_embs, labels=call_labels, title=f"UMAP — {BEATS_MODEL} (last layer) — static"
)
display(fig_umap_beats_static)

In [ ]:
fig_beats_all = plot_umap(
    beats_all_embs, labels=call_labels, title=f"UMAP — {BEATS_MODEL} (all layers avg)", hover_text=hover
)
fig_beats_all.show()
fig_umap_beats_all_static = plot_umap_static(
    beats_all_embs, labels=call_labels, title=f"UMAP — {BEATS_MODEL} (all layers avg) — static"
)
display(fig_umap_beats_all_static)

In [ ]:
fig_effnet = plot_umap(effnet_embs, labels=call_labels, title=f"UMAP — {EFFNET_MODEL}", hover_text=hover)
fig_effnet.show()
fig_umap_effnet_static = plot_umap_static(effnet_embs, labels=call_labels, title=f"UMAP — {EFFNET_MODEL} — static")
display(fig_umap_effnet_static)

## 5. Training-Free Metrics

In [ ]:
print("Computing training-free metrics (NMI, ARI, R-AUC) ...")
print("These evaluate embedding quality without fitting any classifier.\n")

_labels_for_metrics = call_labels

_metric_models = [
    (f"{BEATS_MODEL} (last layer)", beats_embs),
    (f"{BEATS_MODEL} (all layers avg)", beats_all_embs),
    (EFFNET_MODEL, effnet_embs),
]

_metric_rows = []
for _name, _embs in _metric_models:
    _m = compute_training_free_metrics(_embs, _labels_for_metrics)
    _metric_rows.append(
        {"Model": _name, "NMI": round(_m["nmi"], 3), "ARI": round(_m["ari"], 3), "R-AUC": round(_m["r_auc"], 3)}
    )
    print(f"  {_name}: NMI={_m['nmi']:.3f}  ARI={_m['ari']:.3f}  R-AUC={_m['r_auc']:.3f}")

_metrics_df = pd.DataFrame(_metric_rows).set_index("Model")
display(_metrics_df)

## 6. Linear Probe

Two tasks: 11-class call type and 2-class age group.

In [ ]:
probe_kwargs = dict(test_size=0.2, random_state=42, max_iter=1000)
age_labels = df["age"].tolist()

probe_results = {}
for emb_name, embs in [
    (f"{BEATS_MODEL} (last layer)", beats_embs),
    (f"{BEATS_MODEL} (all layers avg)", beats_all_embs),
    (EFFNET_MODEL, effnet_embs),
]:
    for task, task_labels in [("call_type", call_labels), ("age_group", age_labels)]:
        key = f"{emb_name} | {task}"
        res = run_linear_probe(embs, task_labels, **probe_kwargs)
        probe_results[key] = res
        print(f"{key}: accuracy = {res['accuracy']:.3f}")

In [ ]:
rows = [
    {"Model": k.split(" | ")[0], "Task": k.split(" | ")[1], "Accuracy": round(v["accuracy"], 4)}
    for k, v in probe_results.items()
]
acc_df = pd.DataFrame(rows)
display(acc_df.pivot(index="Model", columns="Task", values="Accuracy").style.format("{:.4f}"))

fig_cmp = plot_model_comparison(
    {k: v["accuracy"] for k, v in probe_results.items()},
    title="Zebra Finch — call type & age classification accuracy",
)
fig_cmp.show()
display(
    plot_model_comparison_static(
        {k: v["accuracy"] for k, v in probe_results.items()},
        title="Zebra Finch — linear probe (static)",
    )
)

## 7. Attention Probe (sl-BEATs)

Attention probe on mean-pooled BEATs embeddings for call type and age tasks.

In [ ]:
attn_probe_kwargs = dict(num_heads=8, num_attn_layers=2, epochs=50, test_size=0.2, random_state=42)
attention_results = {}
for emb_name, embs in [
    (f"{BEATS_MODEL} (last layer)", beats_embs),
    (f"{BEATS_MODEL} (all layers avg)", beats_all_embs),
]:
    for task, task_labels in [("call_type", call_labels), ("age_group", age_labels)]:
        key = f"{emb_name} | {task}"
        res = run_attention_probe(embs, task_labels, **attn_probe_kwargs)
        attention_results[key] = res
        print(f"{key} (attention): accuracy = {res['accuracy']:.3f}")

cmp_call = {}
for base in [f"{BEATS_MODEL} (last layer)", f"{BEATS_MODEL} (all layers avg)"]:
    lk = f"{base} | call_type"
    cmp_call[f"{base} (linear)"] = probe_results[lk]["accuracy"]
    cmp_call[f"{base} (attention)"] = attention_results[lk]["accuracy"]
fig_attn_call = plot_model_comparison(cmp_call, title="Zebra Finch — sl-BEATs call type: linear vs attention")
fig_attn_call.show()
display(plot_model_comparison_static(cmp_call, title="Zebra Finch — sl-BEATs call type (static)"))

cmp_age = {}
for base in [f"{BEATS_MODEL} (last layer)", f"{BEATS_MODEL} (all layers avg)"]:
    lk = f"{base} | age_group"
    cmp_age[f"{base} (linear)"] = probe_results[lk]["accuracy"]
    cmp_age[f"{base} (attention)"] = attention_results[lk]["accuracy"]
fig_attn_age = plot_model_comparison(cmp_age, title="Zebra Finch — sl-BEATs age: linear vs attention")
fig_attn_age.show()
display(plot_model_comparison_static(cmp_age, title="Zebra Finch — sl-BEATs age (static)"))

In [ ]:
def plot_confusion_matrix(cm, classes, title):
    cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)
    return px.imshow(
        cm_norm,
        x=classes,
        y=classes,
        color_continuous_scale="Blues",
        zmin=0,
        zmax=1,
        text_auto=".2f",
        title=title,
        labels={"x": "Predicted", "y": "True", "color": "Recall"},
        aspect="auto",
    )


for key in [k for k in probe_results if "call_type" in k]:
    res = probe_results[key]
    plot_confusion_matrix(res["confusion_matrix"], res["classes"], title=f"Call type confusion — {key}").show()
    display(
        confusion_heatmap_static(
            res["confusion_matrix"],
            res["classes"],
            title=f"Call type confusion — {key} (static)",
        )
    )

## 8. Save Artifacts

In [ ]:
ARTIFACTS_DIR = EXAMPLE_DIR / "artifacts"
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

fig_beats.write_html(str(ARTIFACTS_DIR / "umap_beats.html"))
fig_beats_all.write_html(str(ARTIFACTS_DIR / "umap_beats_all_layers.html"))
fig_effnet.write_html(str(ARTIFACTS_DIR / "umap_effnet.html"))
fig_cmp.write_html(str(ARTIFACTS_DIR / "model_comparison.html"))
fig_attn_call.write_html(str(ARTIFACTS_DIR / "model_comparison_beats_attn_call_type.html"))
fig_attn_age.write_html(str(ARTIFACTS_DIR / "model_comparison_beats_attn_age.html"))

fig_umap_beats_static.savefig(str(ARTIFACTS_DIR / "umap_beats_static.png"), dpi=150, bbox_inches="tight")
fig_umap_beats_all_static.savefig(str(ARTIFACTS_DIR / "umap_beats_all_layers_static.png"), dpi=150, bbox_inches="tight")
fig_umap_effnet_static.savefig(str(ARTIFACTS_DIR / "umap_effnet_static.png"), dpi=150, bbox_inches="tight")
plt.close("all")

_metrics_out = {
    "n_calls": len(df),
    "n_call_types": df["call_name"].nunique(),
    "training_free": {
        k: compute_training_free_metrics(v, call_labels)
        for k, v in [("beats_last", beats_embs), ("beats_all_layers", beats_all_embs), ("effnet", effnet_embs)]
    },
    "linear_probe_accuracy": {k: round(v["accuracy"], 4) for k, v in probe_results.items()},
    "attention_probe_accuracy": {k: round(v["accuracy"], 4) for k, v in attention_results.items()},
}
with open(ARTIFACTS_DIR / "zebra_finch_metrics.json", "w") as _f:
    json.dump(_metrics_out, _f, indent=2)
print(f"Artifacts saved to {ARTIFACTS_DIR}")